# Пример запуска: задача про монеты Туг-туг

Ноутбук показывает полный локальный сценарий: создать файл задачи, проверить секреты, запустить выбранные модели через `runner.py` и прочитать ответы из JSON-лога.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'runner.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(ROOT)

/Users/matvei/Documents/projects/random/2026_06_02_cs_space_neiro_solutions


## 1. Создать файл задачи

Задача будет записана в `data/competitions/local_examples/tug_tug_500.json`.

In [2]:
problem = {
    'id': 'tug_tug_500',
    'schema_version': 1,
    'number': 2,
    'title': 'Монеты страны Туг-туг',
    'statement': (
        'В стране Туг-туг в ходу монеты, номиналы которых равны 1, 2 и 3 тугрика. '
        'Однажды Нетуг, житель страны Туг-туг, нашел у себя в кармане 500 монет '
        'суммарным номиналом 1000 тугриков. Докажите, что он может набрать '
        '500 тугриков без сдачи.'
    ),
    'answer': None,
    'solution': None,
    'tags': [],
    'metadata': {},
    'source': 'User-provided olympiad problem',
}

problem_path = Path('data/competitions/local_examples/tug_tug_500.json')
problem_path.parent.mkdir(parents=True, exist_ok=True)
problem_path.write_text(json.dumps(problem, ensure_ascii=False, indent=2), encoding='utf-8')
print(problem_path)
print(problem['statement'])

data/competitions/local_examples/tug_tug_500.json
В стране Туг-туг в ходу монеты, номиналы которых равны 1, 2 и 3 тугрика. Однажды Нетуг, житель страны Туг-туг, нашел у себя в кармане 500 монет суммарным номиналом 1000 тугриков. Докажите, что он может набрать 500 тугриков без сдачи.


## 2. Выбрать модели

По умолчанию запускаются `gpt,gigachat,yandexgpt`. DeepSeek можно добавить в `MODELS` как `deepseek`, когда заполнен `models/deepseek/secrets/.env`. `RUN_ID` создаётся заново на каждый запуск из даты-времени и git hash, чтобы ноутбук не читал старый лог.

In [3]:
from datetime import UTC, datetime
import os
import subprocess

MODEL_ENV_VARS = ('OPENAI_MODEL', 'ANTHROPIC_MODEL', 'GIGACHAT_MODEL', 'YANDEX_MODEL')
for name in MODEL_ENV_VARS:
    os.environ.pop(name, None)

MODELS = 'gpt,gigachat,yandexgpt'  # add ',deepseek' after DEEPSEEK_API_KEY is configured
_git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True)
GIT_HASH = _git.stdout.strip() if _git.returncode == 0 else 'nogit'
RUN_ID = f"tug_tug_{datetime.now(UTC).strftime('%Y%m%d_%H%M%S')}_{GIT_HASH}"
print('models:', MODELS)
print('run_id:', RUN_ID)


models: gpt,gigachat,yandexgpt
run_id: tug_tug_20260603_130247_801b26f


## 2.1. Посмотреть фактически выбранные версии

Эта ячейка показывает, какие версии возьмут адаптеры после загрузки `.env`, `secrets` и `config/models.env`.


In [4]:
import os
from runner import load_env
from models.gpt import GPTModel
from models.gigachat import GigaChatModel
from models.yandexgpt import YandexGPTModel
from models.deepseek import DeepSeekModel

for name in ('OPENAI_MODEL', 'ANTHROPIC_MODEL', 'GIGACHAT_MODEL', 'YANDEX_MODEL'):
    os.environ.pop(name, None)
load_env()
for model in [GPTModel(), GigaChatModel(), YandexGPTModel(), DeepSeekModel()]:
    print(model.__class__.__name__, '->', model.model_id)


GPTModel -> gpt-5.5
GigaChatModel -> GigaChat-2-Max
YandexGPTModel -> yandexgpt-5-pro/latest
DeepSeekModel -> deepseek-v4-pro


## 3. Проверить секреты

Команда не печатает значения ключей.

In [5]:
!python scripts/check_secrets.py --models $MODELS

gpt: ok
gigachat: ok
yandexgpt: ok


## 4. Запустить модели

Основной интерфейс проекта - `runner.py`.

In [6]:
!python runner.py --problem data/competitions/local_examples/tug_tug_500.json --models $MODELS --run-id $RUN_ID

| model                  |   tokens |   cost_usd |   latency_ms | status   | error   |
|------------------------|----------|------------|--------------|----------|---------|
| gpt-5.5                |     1083 |   0.009067 |        18736 | ok       |         |
| GigaChat-2-Max         |      730 |   0        |        17088 | ok       |         |
| yandexgpt-5-pro/latest |      568 |   0.005049 |         5426 | ok       |         |

Run ID: tug_tug_20260603_130247_801b26f
Timestamp: 2026-06-03T13:02:48.212100Z
Git hash: 801b26f
Log written: logs/tug_tug_20260603_130247_801b26f.json


## 5. Прочитать лог и ответы

Ниже выводятся только полезные поля. Полный raw response остается в JSON-логе.

In [7]:
log_path = Path('logs') / f'{RUN_ID}.json'
log = json.loads(log_path.read_text(encoding='utf-8'))
print('run_id:', log.get('run_id'))
print('timestamp:', log.get('timestamp'))
print('git_hash:', log.get('git_hash'))
print('log_path:', log_path)

for result in log['results']:
    print('=' * 80)
    print('model:', result['model'])
    print('status:', 'error' if result['error'] else 'ok')
    print('tokens:', result['prompt_tokens'] + result['completion_tokens'])
    print('cost_usd:', result['cost_usd'])
    print('latency_ms:', result['latency_ms'])
    if result['error']:
        print('error:', result['error'])
    else:
        print(result['answer'])

run_id: tug_tug_20260603_130247_801b26f
timestamp: 2026-06-03T13:02:48.212100Z
git_hash: 801b26f
log_path: logs/tug_tug_20260603_130247_801b26f.json
model: gpt-5.5
status: ok
tokens: 1083
cost_usd: 0.0090675
latency_ms: 18736
Пусть у Нетуга:

- \(a\) монет по 1 тугрику,
- \(b\) монет по 2 тугрика,
- \(c\) монет по 3 тугрика.

Тогда по условию

\[
a+b+c=500
\]

и

\[
a+2b+3c=1000.
\]

Вычтем из второго равенства удвоенное первое:

\[
(a+2b+3c)-2(a+b+c)=1000-1000.
\]

Получаем

\[
-a+c=0,
\]

то есть

\[
a=c.
\]

Значит, количество монет по 1 тугрику равно количеству монет по 3 тугрика. Тогда

\[
a+b+a=500,
\]

то есть

\[
b=500-2a.
\]

Теперь рассмотрим два случая.

### Случай 1: \(a\le 125\)

Тогда

\[
b=500-2a\ge 500-250=250.
\]

Значит, у Нетуга есть как минимум 250 монет по 2 тугрика. Он может взять ровно 250 таких монет. Их сумма равна

\[
250\cdot 2=500.
\]

### Случай 2: \(a\ge 125\)

Так как \(a=c\), у Нетуга есть как минимум 125 монет по 1 тугрику и как минимум 125 монет по 3 т

## 6. Открыть ручной скоринг

После запуска можно открыть веб-интерфейс и оценить ответы.

```bash
python scoring/app.py
```

Затем открыть URL с текущим `RUN_ID`, например `http://127.0.0.1:8000/run/<RUN_ID>`.